In [1]:
"""
Validation of Curvature Implementations
=========================================
Tests against known analytical results on small graphs
where curvatures can be computed by hand.

References:
- Ollivier (2009), Ricci curvature of Markov chains on metric spaces
- Sreejith et al. (2016), Forman curvature for complex networks
- Ni et al. (2019), Community detection on networks with Ricci flow
- Lin, Lu & Yau (2011), Ricci curvature of graphs

Author: D. Sierra-Porta / Claude (validation)
"""

import numpy as np
import networkx as nx
from scipy.optimize import linprog
import sys

PASS = 0
FAIL = 0
WARN = 0


def report(test_name, passed, expected, got, tol=None):
    global PASS, FAIL
    status = "PASS" if passed else "FAIL"
    if passed:
        PASS += 1
    else:
        FAIL += 1
    
    tol_str = f" (tol={tol})" if tol else ""
    print(f"  [{status}] {test_name}: expected={expected}, got={got}{tol_str}")


def warn(test_name, msg):
    global WARN
    WARN += 1
    print(f"  [WARN] {test_name}: {msg}")


# ============================================================
# Import the implementations we want to test
# ============================================================

# Inline copies to make this script self-contained
def forman_ricci_local(G):
    edge_curvatures = {}
    for u, v in G.edges():
        neighbors_u = set(G.neighbors(u)) - {v}
        neighbors_v = set(G.neighbors(v)) - {u}
        common = len(neighbors_u & neighbors_v)
        non_common = len(neighbors_u.symmetric_difference(neighbors_v))
        edge_curvatures[(u, v)] = common + 2 - non_common
    
    node_curvatures = {}
    for n in G.nodes():
        adj_curvs = []
        for u, v in G.edges(n):
            key = (u, v) if (u, v) in edge_curvatures else (v, u)
            if key in edge_curvatures:
                adj_curvs.append(edge_curvatures[key])
        node_curvatures[n] = np.mean(adj_curvs) if adj_curvs else 0.0
    
    return edge_curvatures, node_curvatures


def _wasserstein_1d(mu, nu, cost_matrix):
    n = len(mu)
    m = len(nu)
    c = cost_matrix.flatten()
    A_eq = np.zeros((n + m, n * m))
    for i in range(n):
        A_eq[i, i * m:(i + 1) * m] = 1
    for j in range(m):
        A_eq[n + j, j::m] = 1
    b_eq = np.concatenate([mu, nu])
    result = linprog(c, A_eq=A_eq, b_eq=b_eq,
                     bounds=[(0, None)] * (n * m),
                     method='highs', options={'presolve': True})
    return result.fun if result.success else np.nan


def ollivier_ricci_edge(G, u, v, alpha=0.5):
    """Compute OR curvature for a single edge (u,v)."""
    nodes = list(G.nodes())
    node_idx = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    
    dist_dict = dict(nx.all_pairs_shortest_path_length(G))
    dist_matrix = np.full((n, n), np.inf)
    for a in nodes:
        for b, d in dist_dict[a].items():
            dist_matrix[node_idx[a]][node_idx[b]] = d
    
    neighbors_u = list(G.neighbors(u))
    support_u = [u] + neighbors_u
    mu = np.zeros(len(support_u))
    mu[0] = alpha
    if neighbors_u:
        mu[1:] = (1 - alpha) / len(neighbors_u)
    else:
        mu[0] = 1.0
    
    neighbors_v = list(G.neighbors(v))
    support_v = [v] + neighbors_v
    nu = np.zeros(len(support_v))
    nu[0] = alpha
    if neighbors_v:
        nu[1:] = (1 - alpha) / len(neighbors_v)
    else:
        nu[0] = 1.0
    
    cost = np.zeros((len(support_u), len(support_v)))
    for i, su in enumerate(support_u):
        for j, sv in enumerate(support_v):
            cost[i, j] = dist_matrix[node_idx[su]][node_idx[sv]]
    
    W = _wasserstein_1d(mu, nu, cost)
    d_uv = dist_matrix[node_idx[u]][node_idx[v]]
    
    if d_uv > 0 and not np.isnan(W):
        return 1.0 - W / d_uv
    return 0.0


# ============================================================
# TEST 1: FORMAN-RICCI ON COMPLETE GRAPHS
# ============================================================

def test_forman_complete_graph():
    """
    For a complete graph K_n, every edge (u,v) has:
    - neighbors_u \ {v} = n-2 nodes
    - neighbors_v \ {u} = n-2 nodes  
    - common neighbors = n-2 (all remaining nodes)
    - non-common neighbors = 0 (symmetric difference is empty)
    - F(u,v) = (n-2) + 2 - 0 = n
    
    So for K_4: F = 4, K_5: F = 5, etc.
    """
    print("\nTEST 1: Forman-Ricci on Complete Graphs")
    print("-" * 50)
    
    for n in [3, 4, 5, 6, 10]:
        G = nx.complete_graph(n)
        edge_curv, node_curv = forman_ricci_local(G)
        
        expected = n
        values = list(edge_curv.values())
        
        all_equal = all(abs(v - expected) < 1e-10 for v in values)
        report(f"K_{n}: all edges F={expected}", all_equal, expected, values[0])


def test_forman_path_graph():
    """
    For a path graph P_n (0-1-2-...-n-1):
    - Interior edge (i, i+1) where both have degree 2:
      neighbors_i \ {i+1} = {i-1}, neighbors_{i+1} \ {i} = {i+2}
      common = 0, non_common = 2
      F = 0 + 2 - 2 = 0
    
    - Boundary edge (0,1): 
      neighbors_0 \ {1} = {} (degree 1), neighbors_1 \ {0} = {2}
      common = 0, non_common = 1
      F = 0 + 2 - 1 = 1
    
    Similarly for the other boundary edge.
    """
    print("\nTEST 2: Forman-Ricci on Path Graphs")
    print("-" * 50)
    
    # Path with 5 nodes: 0-1-2-3-4
    G = nx.path_graph(5)
    edge_curv, _ = forman_ricci_local(G)
    
    # Boundary edges
    report("P_5 edge (0,1)", abs(edge_curv[(0, 1)] - 1) < 1e-10, 1, edge_curv[(0, 1)])
    report("P_5 edge (3,4)", abs(edge_curv[(3, 4)] - 1) < 1e-10, 1, edge_curv[(3, 4)])
    
    # Interior edges
    report("P_5 edge (1,2)", abs(edge_curv[(1, 2)] - 0) < 1e-10, 0, edge_curv[(1, 2)])
    report("P_5 edge (2,3)", abs(edge_curv[(2, 3)] - 0) < 1e-10, 0, edge_curv[(2, 3)])


def test_forman_cycle_graph():
    """
    For a cycle graph C_n, every node has degree 2.
    For any edge (u,v):
    - neighbors_u \ {v} = 1 node
    - neighbors_v \ {u} = 1 node
    - For n >= 5: these are different nodes, so common = 0, non_common = 2
      F = 0 + 2 - 2 = 0
    - For n = 4: neighbors are different, common = 0, non_common = 2, F = 0
    - For n = 3 (triangle): the single neighbor of each is the same (the third node)
      common = 1, non_common = 0, F = 1 + 2 - 0 = 3
    """
    print("\nTEST 3: Forman-Ricci on Cycle Graphs")
    print("-" * 50)
    
    # Triangle C_3
    G = nx.cycle_graph(3)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("C_3: all edges F=3", abs(val - 3) < 1e-10, 3, val)
    
    # Square C_4
    G = nx.cycle_graph(4)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("C_4: all edges F=0", abs(val - 0) < 1e-10, 0, val)
    
    # Pentagon C_5
    G = nx.cycle_graph(5)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("C_5: all edges F=0", abs(val - 0) < 1e-10, 0, val)
    
    # C_6
    G = nx.cycle_graph(6)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("C_6: all edges F=0", abs(val - 0) < 1e-10, 0, val)


def test_forman_star_graph():
    """
    Star graph S_n: central node 0 connected to nodes 1, 2, ..., n.
    
    For edge (0, i):
    - neighbors_0 \ {i} = {1,...,n} \ {i} = n-1 nodes
    - neighbors_i \ {0} = {} (leaf, degree 1)
    - common = 0
    - non_common = n-1
    - F = 0 + 2 - (n-1) = 3 - n
    
    Star with 4 leaves (S_4): F = 3 - 4 = -1
    Star with 3 leaves (S_3): F = 3 - 3 = 0
    """
    print("\nTEST 4: Forman-Ricci on Star Graphs")
    print("-" * 50)
    
    # S_3 (center + 3 leaves)
    G = nx.star_graph(3)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("S_3: F=0", abs(val - 0) < 1e-10, 0, val)
    
    # S_4
    G = nx.star_graph(4)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("S_4: F=-1", abs(val - (-1)) < 1e-10, -1, val)
    
    # S_5
    G = nx.star_graph(5)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("S_5: F=-2", abs(val - (-2)) < 1e-10, -2, val)
    
    # S_10
    G = nx.star_graph(10)
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("S_10: F=-7", abs(val - (-7)) < 1e-10, -7, val)


# ============================================================
# TEST 5: OLLIVIER-RICCI ON KNOWN CASES
# ============================================================

def test_ollivier_complete_graph():
    """
    For complete graph K_n with alpha=0:
    
    The lazy random walk measure at node x with alpha=0 is:
    m_x(v) = 1/(n-1) for all neighbors v of x
    
    For K_n, every node is neighbor of every other.
    mu_x and mu_y are uniform over all nodes except themselves.
    
    For alpha=0 on K_n, the known result (Ollivier 2009, Lin-Lu-Yau 2011) is:
    kappa(x,y) = 2/n  for adjacent vertices in K_n
    
    For alpha=0.5, the computation changes. Let's verify with small cases.
    
    For K_3 with alpha=0:
    - mu_x: {y: 0.5, z: 0.5}
    - mu_y: {x: 0.5, z: 0.5}  
    - Wasserstein: optimal transport sends x->y (cost 1, mass 0.5) and z->z (cost 0, mass 0.5)
    - W = 0.5 * 1 + 0.5 * 0 = 0.5
    - kappa = 1 - 0.5/1 = 0.5
    
    Known: kappa(K_3, alpha=0) = 0.5  (matches 2/n = 2/3? No, 2/3 for different definition)
    
    Actually for Lin-Lu-Yau definition (alpha=0):
    On K_n: kappa = 2/(n-1) - but this depends on the exact definition.
    
    Let me just verify K_3 by hand with alpha=0.
    """
    print("\nTEST 5: Ollivier-Ricci on Complete Graphs")
    print("-" * 50)
    
    # K_3 with alpha=0
    G = nx.complete_graph(3)
    kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
    # By hand: mu_0 = {1: 0.5, 2: 0.5}, mu_1 = {0: 0.5, 2: 0.5}
    # Optimal: send mass 0.5 from node 2 to node 2 (cost 0), 
    #          send mass 0.5 from node 1 to node 0 (cost 1)
    # W = 0.5, d=1, kappa = 1 - 0.5 = 0.5
    report("K_3 (α=0): κ=0.5", abs(kappa - 0.5) < 0.01, 0.5, round(kappa, 4), tol=0.01)
    
    # K_4 with alpha=0
    G = nx.complete_graph(4)
    kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
    # mu_0 = {1: 1/3, 2: 1/3, 3: 1/3}, mu_1 = {0: 1/3, 2: 1/3, 3: 1/3}
    # Optimal: 2->2 (0), 3->3 (0), 1->0 (cost 1, mass 1/3)
    # W = 1/3, kappa = 1 - 1/3 = 2/3
    expected = 2.0 / 3.0
    report("K_4 (α=0): κ=2/3", abs(kappa - expected) < 0.01, round(expected, 4), round(kappa, 4), tol=0.01)
    
    # K_5 with alpha=0
    G = nx.complete_graph(5)
    kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
    # mu_0 = {1:1/4, 2:1/4, 3:1/4, 4:1/4}, mu_1 = {0:1/4, 2:1/4, 3:1/4, 4:1/4}
    # Optimal: 2->2, 3->3, 4->4 (cost 0 each), 1->0 (cost 1, mass 1/4)
    # W = 1/4, kappa = 1 - 1/4 = 3/4
    expected = 3.0 / 4.0
    report("K_5 (α=0): κ=3/4", abs(kappa - expected) < 0.01, round(expected, 4), round(kappa, 4), tol=0.01)
    
    # General pattern: K_n with alpha=0 -> kappa = (n-2)/(n-1)
    print("  --- General pattern K_n (α=0): κ = (n-2)/(n-1) ---")
    for n in [6, 8, 10]:
        G = nx.complete_graph(n)
        kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
        expected = (n - 2) / (n - 1)
        report(f"K_{n} (α=0): κ={(n-2)}/{(n-1)}", 
               abs(kappa - expected) < 0.01, round(expected, 4), round(kappa, 4), tol=0.01)


def test_ollivier_path_graph():
    """
    Path graph with alpha=0.
    
    For interior edge (i, i+1) on a long path:
    - mu_i = {i-1: 0.5, i+1: 0.5}
    - mu_{i+1} = {i: 0.5, i+2: 0.5}
    - Optimal transport: i-1 -> i (cost 1, mass 0.5), i+1 -> i+2 (cost 1, mass 0.5)
    - W = 1.0, d = 1, kappa = 1 - 1 = 0
    
    For boundary edge (0, 1):
    - mu_0 = {1: 1.0} (only neighbor)
    - mu_1 = {0: 0.5, 2: 0.5}
    - Optimal: send 1.0 from node 1. Best: 0.5 to node 0 (cost 1), 0.5 to node 2 (cost 1)
    - Wait, mu_0 has all mass at 1, mu_1 has mass at 0 and 2.
    - Transport 1->0 (0.5 mass, cost 1) and 1->2 (0.5 mass, cost 1)
    - W = 1.0, kappa = 1 - 1 = 0
    
    Hmm, but with alpha=0 and degree-1 node, mu_0(1) = 1.0
    """
    print("\nTEST 6: Ollivier-Ricci on Path Graph (α=0)")
    print("-" * 50)
    
    G = nx.path_graph(7)  # 0-1-2-3-4-5-6
    
    # Interior edge
    kappa_interior = ollivier_ricci_edge(G, 3, 4, alpha=0.0)
    report("P_7 interior (3,4) α=0: κ=0", abs(kappa_interior - 0.0) < 0.05, 0.0, 
           round(kappa_interior, 4), tol=0.05)
    
    # Now with alpha=0.5 (our default)
    # Interior edge (3,4), alpha=0.5:
    # mu_3 = {3: 0.5, 2: 0.25, 4: 0.25}
    # mu_4 = {4: 0.5, 3: 0.25, 5: 0.25}
    # Optimal: 3->4 (mass 0.5, cost 1), 2->3 (mass 0.25, cost 1), 4->5 (mass 0.25, cost 1)
    # W = 0.5*1 + 0.25*1 + 0.25*1 = 1.0... but maybe there's a better transport
    # Actually: 3->3 via 4's measure? Let me think...
    # mu_3: support is {2, 3, 4} with probs {0.25, 0.5, 0.25}
    # mu_4: support is {3, 4, 5} with probs {0.25, 0.5, 0.25}
    # Optimal: 3->3 (mass 0.25, cost 0), 3->4 (mass 0.25, cost 1), 
    #          4->4 (mass 0.25, cost 0), 2->5 would be expensive...
    # Better: 2->3 (cost 1, mass 0.25), 3->3 (cost 0, mass 0.25), 3->4 (cost 1, mass 0.25), 4->5 (cost 1, mass 0.25)
    # Hmm this is getting complex. Let's just check it runs and gives reasonable value.
    
    kappa_interior_05 = ollivier_ricci_edge(G, 3, 4, alpha=0.5)
    reasonable = -1.0 < kappa_interior_05 < 1.0
    report("P_7 interior (3,4) α=0.5: reasonable range", reasonable, 
           "(-1, 1)", round(kappa_interior_05, 4))


def test_ollivier_cycle_graph():
    """
    For cycle C_n with alpha=0:
    Every edge has the same curvature by symmetry.
    
    For C_n (n >= 4), alpha=0:
    - mu_x = uniform on two neighbors
    - mu_y = uniform on two neighbors
    - The two distributions share no support (for n >= 5)
    - For n >= 5: kappa = 0 (same as path interior)
    
    For C_4 (alpha=0):
    - mu_0 = {1: 0.5, 3: 0.5}
    - mu_1 = {0: 0.5, 2: 0.5}
    - Transport: 1->0 (cost 1, mass 0.5), 3->2 (cost 1, mass 0.5)
    - W = 1.0, kappa = 0
    
    For C_3 (alpha=0): same as K_3, kappa = 0.5
    """
    print("\nTEST 7: Ollivier-Ricci on Cycle Graphs (α=0)")
    print("-" * 50)
    
    # C_3 = K_3
    G = nx.cycle_graph(3)
    kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
    report("C_3 (α=0): κ=0.5", abs(kappa - 0.5) < 0.01, 0.5, round(kappa, 4), tol=0.01)
    
    # C_4
    G = nx.cycle_graph(4)
    kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
    report("C_4 (α=0): κ=0", abs(kappa - 0.0) < 0.05, 0.0, round(kappa, 4), tol=0.05)
    
    # C_6
    G = nx.cycle_graph(6)
    kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
    report("C_6 (α=0): κ=0", abs(kappa - 0.0) < 0.05, 0.0, round(kappa, 4), tol=0.05)


def test_ollivier_star_graph():
    """
    Star graph S_n, edge from center (0) to leaf (1), alpha=0.
    
    - mu_0 = uniform on {1, 2, ..., n} -> each gets 1/n
    - mu_1 = {0: 1.0} (only neighbor is center)
    
    Transport from mu_0 to mu_1:
    - All mass of mu_1 is at node 0. Mass of mu_0 is at leaves.
    - Move 1/n from leaf 1 to node 0: cost 1 (distance 1)
    - Move 1/n from each other leaf i to node 0: cost 1 each
    - Total W = n * (1/n) * 1 = 1.0
    - kappa = 1 - 1/1 = 0
    
    Wait, that's not right. Let me reconsider.
    mu_0: support = {1, 2, ..., n}, each with mass 1/n
    mu_1: support = {0}, mass = 1.0
    
    Transport: move all mass from leaves to node 0. Each leaf is distance 1 from node 0.
    W = sum over leaves: (1/n) * 1 = 1
    kappa = 1 - 1/1 = 0
    
    But actually with alpha=0, mu_1 = {0: 1} means all mass at node 0 (center).
    So mu_0 has mass spread on leaves, mu_1 has mass at center.
    Transporting from each leaf to center costs 1 each.
    W = 1, d(0,1) = 1, kappa = 0.
    """
    print("\nTEST 8: Ollivier-Ricci on Star Graphs (α=0)")
    print("-" * 50)
    
    for n in [3, 4, 5]:
        G = nx.star_graph(n)
        kappa = ollivier_ricci_edge(G, 0, 1, alpha=0.0)
        report(f"S_{n} (α=0) center-leaf: κ=0", abs(kappa - 0.0) < 0.05, 0.0, 
               round(kappa, 4), tol=0.05)


# ============================================================
# TEST 9: CONSISTENCY CHECKS
# ============================================================

def test_symmetry():
    """
    Curvature on undirected graphs should be symmetric:
    κ(u,v) = κ(v,u) for both OR and FR.
    """
    print("\nTEST 9: Symmetry Check")
    print("-" * 50)
    
    G = nx.petersen_graph()
    
    # Forman
    edge_curv, _ = forman_ricci_local(G)
    symmetric = True
    for u, v in G.edges():
        fr_uv = edge_curv.get((u, v), edge_curv.get((v, u), None))
        # Since we only store (u,v) not (v,u), this is inherently symmetric
        # But let's check the computation is consistent
        
        neighbors_u = set(G.neighbors(u)) - {v}
        neighbors_v = set(G.neighbors(v)) - {u}
        common = len(neighbors_u & neighbors_v)
        non_common = len(neighbors_u.symmetric_difference(neighbors_v))
        fr1 = common + 2 - non_common
        
        # Swap u and v
        neighbors_v2 = set(G.neighbors(v)) - {u}
        neighbors_u2 = set(G.neighbors(u)) - {v}
        common2 = len(neighbors_v2 & neighbors_u2)
        non_common2 = len(neighbors_v2.symmetric_difference(neighbors_u2))
        fr2 = common2 + 2 - non_common2
        
        if abs(fr1 - fr2) > 1e-10:
            symmetric = False
            break
    
    report("FR symmetry (Petersen)", symmetric, "symmetric", "symmetric" if symmetric else "asymmetric")
    
    # Ollivier (check a few edges)
    edges_to_check = list(G.edges())[:5]
    or_symmetric = True
    max_diff = 0
    for u, v in edges_to_check:
        k_uv = ollivier_ricci_edge(G, u, v, alpha=0.5)
        k_vu = ollivier_ricci_edge(G, v, u, alpha=0.5)
        diff = abs(k_uv - k_vu)
        max_diff = max(max_diff, diff)
        if diff > 0.01:
            or_symmetric = False
    
    report("OR symmetry (Petersen, 5 edges)", or_symmetric, 
           "symmetric", f"max_diff={max_diff:.6f}", tol=0.01)


def test_regularity():
    """
    On regular graphs, all edges should have the same curvature
    for both FR and OR (by symmetry of the graph automorphism group).
    
    Test on Petersen graph (3-regular) and complete graphs.
    """
    print("\nTEST 10: Regularity Check (uniform curvature on vertex-transitive graphs)")
    print("-" * 50)
    
    # Petersen graph (vertex-transitive, 3-regular)
    G = nx.petersen_graph()
    edge_curv, _ = forman_ricci_local(G)
    vals = list(edge_curv.values())
    all_same = all(abs(v - vals[0]) < 1e-10 for v in vals)
    report("FR: Petersen graph (all edges equal)", all_same, vals[0], 
           f"range=[{min(vals):.2f}, {max(vals):.2f}]")
    
    # Check a few OR values on Petersen
    edges = list(G.edges())[:5]
    or_vals = [ollivier_ricci_edge(G, u, v, alpha=0.0) for u, v in edges]
    or_std = np.std(or_vals)
    report("OR: Petersen graph (edges ~equal, α=0)", or_std < 0.01,
           "std≈0", f"std={or_std:.6f}, vals={[round(v,4) for v in or_vals]}", tol=0.01)


def test_visibility_graph_basic():
    """
    Test that VG construction + curvature pipeline works on known sequences.
    
    Monotonic increasing sequence: each node sees all subsequent nodes.
    VG should be a complete graph for strictly increasing sequences.
    """
    print("\nTEST 11: Visibility Graph + Curvature Pipeline")
    print("-" * 50)
    
    # Strictly increasing -> complete graph
    data = [1, 2, 3, 4, 5]
    G = nx.visibility_graph(data)
    n = len(data)
    expected_edges = n * (n - 1) // 2
    report(f"VG(increasing): K_{n}", G.number_of_edges() == expected_edges,
           expected_edges, G.number_of_edges())
    
    # For K_5, FR should be 5 on every edge
    edge_curv, _ = forman_ricci_local(G)
    val = list(edge_curv.values())[0]
    report("VG(increasing) FR: all edges = 5", abs(val - 5) < 1e-10, 5, val)
    
    # V-shape: [5, 1, 5] -> nodes 0 and 2 cannot see each other (blocked by... 
    # actually 1 < 5, so 0 can see 2)
    # Let's use [5, 3, 1, 3, 5] - symmetric
    data2 = [5, 3, 1, 3, 5]
    G2 = nx.visibility_graph(data2)
    report(f"VG([5,3,1,3,5]): n=5, edges={G2.number_of_edges()}", 
           G2.number_of_nodes() == 5, 5, G2.number_of_nodes())
    
    # Just check FR runs without error
    edge_curv2, node_curv2 = forman_ricci_local(G2)
    report("VG([5,3,1,3,5]) FR: runs OK", len(node_curv2) == 5, 5, len(node_curv2))
    
    # Random series
    np.random.seed(42)
    data3 = np.random.normal(0, 1, 50)
    G3 = nx.visibility_graph(data3)
    edge_curv3, node_curv3 = forman_ricci_local(G3)
    report(f"VG(random, n=50): FR computed for all nodes", 
           len(node_curv3) == 50, 50, len(node_curv3))


def test_wasserstein_known():
    """
    Test the Wasserstein distance computation directly.
    
    W_1 between delta_0 and delta_1 on {0, 1} with cost d(0,1)=1:
    W = 1.0
    
    W_1 between uniform(0,1) and delta_0 on {0, 1}:
    mu = [0.5, 0.5], nu = [1.0, 0.0], cost = [[0, 1], [1, 0]]
    W = 0.5 * 0 + 0.5 * 1 = 0.5
    """
    print("\nTEST 12: Wasserstein Distance Computation")
    print("-" * 50)
    
    # Delta to delta
    mu = np.array([1.0, 0.0])
    nu = np.array([0.0, 1.0])
    cost = np.array([[0, 1], [1, 0]], dtype=float)
    W = _wasserstein_1d(mu, nu, cost)
    report("W(δ_0, δ_1) = 1.0", abs(W - 1.0) < 1e-6, 1.0, round(W, 6))
    
    # Uniform to delta
    mu = np.array([0.5, 0.5])
    nu = np.array([1.0, 0.0])
    W = _wasserstein_1d(mu, nu, cost)
    report("W(uniform, δ_0) = 0.5", abs(W - 0.5) < 1e-6, 0.5, round(W, 6))
    
    # Same distribution
    mu = np.array([0.3, 0.7])
    nu = np.array([0.3, 0.7])
    W = _wasserstein_1d(mu, nu, cost)
    report("W(p, p) = 0", abs(W) < 1e-6, 0.0, round(W, 6))
    
    # 3-point space
    mu = np.array([1.0, 0.0, 0.0])
    nu = np.array([0.0, 0.0, 1.0])
    cost = np.array([[0, 1, 2], [1, 0, 1], [2, 1, 0]], dtype=float)
    W = _wasserstein_1d(mu, nu, cost)
    report("W(δ_0, δ_2) on path = 2.0", abs(W - 2.0) < 1e-6, 2.0, round(W, 6))


# ============================================================
# MAIN
# ============================================================

def main():
    global PASS, FAIL, WARN
    
    print("=" * 60)
    print("CURVATURE IMPLEMENTATION VALIDATION SUITE")
    print("=" * 60)
    
    test_forman_complete_graph()
    test_forman_path_graph()
    test_forman_cycle_graph()
    test_forman_star_graph()
    test_wasserstein_known()
    test_ollivier_complete_graph()
    test_ollivier_path_graph()
    test_ollivier_cycle_graph()
    test_ollivier_star_graph()
    test_symmetry()
    test_regularity()
    test_visibility_graph_basic()
    
    print("\n" + "=" * 60)
    print(f"RESULTS: {PASS} passed, {FAIL} failed, {WARN} warnings")
    print("=" * 60)
    
    if FAIL == 0:
        print("\n  ALL TESTS PASSED. Implementation is reliable.")
    else:
        print(f"\n  WARNING: {FAIL} tests failed. Review implementation.")
    
    return FAIL == 0


if __name__ == "__main__":
    success = main()
    sys.exit(0 if success else 1)

<>:136: SyntaxWarning: invalid escape sequence '\ '
<>:162: SyntaxWarning: invalid escape sequence '\ '
<>:193: SyntaxWarning: invalid escape sequence '\ '
<>:234: SyntaxWarning: invalid escape sequence '\ '
<>:136: SyntaxWarning: invalid escape sequence '\ '
<>:162: SyntaxWarning: invalid escape sequence '\ '
<>:193: SyntaxWarning: invalid escape sequence '\ '
<>:234: SyntaxWarning: invalid escape sequence '\ '
/var/folders/8n/h_rtmhz56bd84hgb5_cp68fc0000gn/T/ipykernel_2774/3829722160.py:136: SyntaxWarning: invalid escape sequence '\ '
  - neighbors_u \ {v} = n-2 nodes
/var/folders/8n/h_rtmhz56bd84hgb5_cp68fc0000gn/T/ipykernel_2774/3829722160.py:162: SyntaxWarning: invalid escape sequence '\ '
  neighbors_i \ {i+1} = {i-1}, neighbors_{i+1} \ {i} = {i+2}
/var/folders/8n/h_rtmhz56bd84hgb5_cp68fc0000gn/T/ipykernel_2774/3829722160.py:193: SyntaxWarning: invalid escape sequence '\ '
  - neighbors_u \ {v} = 1 node
/var/folders/8n/h_rtmhz56bd84hgb5_cp68fc0000gn/T/ipykernel_2774/3829722160.py

CURVATURE IMPLEMENTATION VALIDATION SUITE

TEST 1: Forman-Ricci on Complete Graphs
--------------------------------------------------
  [PASS] K_3: all edges F=3: expected=3, got=3
  [PASS] K_4: all edges F=4: expected=4, got=4
  [PASS] K_5: all edges F=5: expected=5, got=5
  [PASS] K_6: all edges F=6: expected=6, got=6
  [PASS] K_10: all edges F=10: expected=10, got=10

TEST 2: Forman-Ricci on Path Graphs
--------------------------------------------------
  [PASS] P_5 edge (0,1): expected=1, got=1
  [PASS] P_5 edge (3,4): expected=1, got=1
  [PASS] P_5 edge (1,2): expected=0, got=0
  [PASS] P_5 edge (2,3): expected=0, got=0

TEST 3: Forman-Ricci on Cycle Graphs
--------------------------------------------------
  [PASS] C_3: all edges F=3: expected=3, got=3
  [PASS] C_4: all edges F=0: expected=0, got=0
  [PASS] C_5: all edges F=0: expected=0, got=0
  [PASS] C_6: all edges F=0: expected=0, got=0

TEST 4: Forman-Ricci on Star Graphs
--------------------------------------------------
  

SystemExit: 1

/opt/homebrew/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
